In [1]:
import sys
sys.path.append("../src")
from utils import *

In [29]:
class TriStateBuffer:
    """
    Simulates a Tri-State Buffer for an 8-bit bus using pure logic gates.
    Instead of outputting 'None' when disabled, it forces the output to all 0s.
    (PengChen:) use if condition and bool variable to simulate this buffer
    so ugly, so gemini give me this implement.
    """
    def __init__(self, nbits=8):
        self.nbits = nbits
        self.and_gates = [AND(2) for _ in range(self.nbits)]

    def __call__(self, data_in, enable):
        # If enable=1, 1 AND Data = Data. (Passes through)
        # If enable=0, 0 AND Data = 0. (Blocks data, masking it to 0s)
        
        output = []
        for i in range(self.nbits):
            output.append(self.and_gates[i]([data_in[i], enable]))
            
        return output

In [30]:
class DataBus:
    """ 
    (PengChen:) be compatible class TriStateBuffer
    written by gemini
    """
    def __init__(self, num_buffers, nbits=8):
        self.nbits = nbits
        self.num_buffers = num_buffers
        # A giant OR gate for each wire on the bus. 
        # If num_buffers is 8, this builds an 8-input OR gate for each wire.
        self.or_gates = [OR(self.num_buffers) for _ in range(self.nbits)] 

    def __call__(self, list_of_buffer_outputs):
        assert len(list_of_buffer_outputs) == self.num_buffers, f"Bus expects {self.num_buffers} connections!"
        
        bus_result = []
        for i in range(self.nbits):
            # This slices vertically! It grabs bit `i` from EVERY buffer in the list.
            # E.g., for wire 0, it grabs bit 0 from Reg A, bit 0 from Reg B, bit 0 from Reg C...
            bits_for_this_wire = [buf_out[i] for buf_out in list_of_buffer_outputs]
            
            # Shove all those bits into the giant OR gate
            bus_result.append(self.or_gates[i](bits_for_this_wire))
            
        return bus_result

In [ ]:
class RegisterArray:
    """
    implement like: https://codehiddenlanguage.com/Chapter22/
    and page on book: 344
    """
    def __init__(self, nbits = 8):
        self.nbits = nbits # bits of latch
        self.decoder_clock = Decoder_3_8()
        self.decoder_enable = Decoder_3_8()

        self.latch_A = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)
        self.latch_B = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)
        self.latch_C = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)
        self.latch_D = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)
        self.latch_E = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)
        self.latch_H = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)
        self.latch_L = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)
        
        self.latchs = { 0: self.latch_B, 
                        1: self.latch_C,
                        2: self.latch_D,
                        3: self.latch_E,
                        4: self.latch_H,
                        5: self.latch_L,
                        # watch out, 6: Memory
                        7: self.latch_A,}
        
        self.tri_A = TriStateBuffer(self.nbits)
        self.tri_B = TriStateBuffer(self.nbits)
        self.tri_C = TriStateBuffer(self.nbits)
        self.tri_D = TriStateBuffer(self.nbits)
        self.tri_E = TriStateBuffer(self.nbits)
        self.tri_H = TriStateBuffer(self.nbits)
        self.tri_L = TriStateBuffer(self.nbits)
        
        self.tris = {   0: self.tri_B, 
                        1: self.tri_C,
                        2: self.tri_D,
                        3: self.tri_E,
                        4: self.tri_H,
                        5: self.tri_L,
                        # watch out, 6: Memory
                        7: self.tri_A,}

        self.data_bus = DataBus(num_buffers=len(self.latchs), nbits=self.nbits)
        self.addr_nbits = 16
        # num_buffers need change after
        self.addr_bus = DataBus(num_buffers=7, nbits=self.addr_nbits)
        self.tri_hl = TriStateBuffer(self.addr_nbits)

        self.clock_idx = [0] * (len(self.latchs) + 1) 
        self.enable_idx = [0] * (len(self.latchs) + 1) 
        self.or_gate_for_acc_clk = OR()
        self.or_gate_for_acc_read = OR()

        # self.inst_latch1 = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)
        # self.inst_latch2 = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)
        # self.enable_inst_latch2 = 0
        # self.inst_latch3 = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)

        # page 352: 
        self.or_gate_h_clk = OR()
        self.or_gate_l_clk = OR()
    
    def __call__(self, data, select, clock, hl_clock, hl_select, addr):
        assert len(data) == self.nbits, f"Inputs must be {self.nbits} bits long"
        assert len(select) == 3, f"selects must be 3 bits long"
        assert clock in [0, 1], "clock signal must be 0 or 1"

        # select is [i0, i1, i2]
        # latch_idx will let latch_idx[i0*4+i1*2+i2] = 1, other is 0
        # example [1, 1, 1]
        # latch_idx is [0, 0, 0, 0, 0, 0, 0, 1]
        # that suppose clock/enable is 1, otherwise latch_idx is all zeros
        clock_idx = self.decoder_clock(select, enable=clock)
        assert (sum(clock_idx) == 1 and clock == 1) or ((sum(clock_idx) == 0 and clock == 0)), "FATAL: Check Decoder_3_8 input/output"
        
        duplicate_datas = self.duplicate_data_to_each_latch(data)
        # clock_idx[3] = self.or_gate_h_clk([clock_idx[3], HL_clock])
        # clock_idx[2] = self.or_gate_l_clk([clock_idx[2], HL_clock])
        
        if hl_select:
            duplicate_datas[4] = addr[:8]
            duplicate_datas[5] = addr[8:]
            clock_idx[4] = self.or_gate_h_clk([clock_idx[4], hl_clock])
            clock_idx[5] = self.or_gate_l_clk([clock_idx[5], hl_clock])
        self.clock_idx = clock_idx

        self.write_helper(duplicate_datas, clock_idx)

    def duplicate_data_to_each_latch(self, data):
        datas = []
        for _ in range(len(self.latchs) + 1):
            datas.append(data)
        return datas

    def write_accumulator(self, data, clk):
        self.clock_idx[7] = self.or_gate_for_acc_clk([clk, self.clock_idx[7]])
        assert sum(self.clock_idx) == 1
        self.write_helper(self.duplicate_data_to_each_latch(data), self.clock_idx)
        
    def write_helper(self, datas, clock_idx):
        for idx, clk in enumerate(clock_idx):
            if idx == 6: # select is [1, 1, 0]
                continue
            self.latchs[idx](datas[idx], clk)

    def read_hl(self, enable_hl):
        return self.tri_hl(self.latch_H.getQ() + self.latch_L.getQ(), enable_hl)

    def read_accumulator(self, enable):
        self.enable_idx[7] = self.or_gate_for_acc_read([enable, self.enable_idx[7]])
        assert sum(self.enable_idx) == 1
        return self.read_helper(self, self.enable_idx)

    def read_register(self, select, enable):
        assert len(select) == 3, f"selects must be 3 bits long"
        assert enable in [0, 1], "enable signal must be 0 or 1"
        
        enable_idx = self.decoder_enable(select, enable=enable)
        assert (sum(enable_idx) == 1 and enable == 1) or ((sum(enable_idx) == 0 and enable == 0)), "FATAL: Check Decoder_3_8 input/output"
        self.enable_idx = enable_idx
        return self.read_helper(self, enable_idx)
    
    def read_helper(self, enable_idx):
        list_of_buffer_outputs = []
        for idx, enab in enumerate(enable_idx):
            if idx == 6: # select is [1, 1, 0]
                continue
            list_of_buffer_outputs.append(self.tris[idx](self.latchs[idx].getQ(), enab))

        return self.data_bus(list_of_buffer_outputs)


In [43]:
x = [0, 1, 4]
y = []
for _ in range(4):
    y.append(x)
y

[[0, 1, 4], [0, 1, 4], [0, 1, 4], [0, 1, 4]]

In [32]:
class InstLatch:
    def __init__(self, nbits = 8):
        self.nbits = nbits
        self.inst_latch1 = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)
        self.inst_latch2 = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)
        self.inst_latch3 = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nbits)

        self.inst_latchs = {
            1:self.inst_latch1,
            2:self.inst_latch2,
            3:self.inst_latch3,
        }

        self.tri_2 = TriStateBuffer(self.nbits)
        
    def __call__(self, *args, **kwds):
        pass
        
    def write_latch1(self, data, clock_idx):
        self.write_helper(data, 1, clock_idx)

    def write_latch2(self, data, clock_idx):
        self.write_helper(data, 2, clock_idx)

    def write_latch3(self, data, clock_idx):
        self.write_helper(data, 3, clock_idx)

    def write_helper(self, data, latch_idx, clock_idx):
        self.inst_latchs[latch_idx](data, clock_idx)

    def read_latch2(self, enable):
        return self.tri_2(self.inst_latch2.getQ(), enable)

In [33]:
class ProgramCounter:
    def __init__(self):
        self.addr_bits = 16
        self.latch = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.addr_bits)
        self.tri = TriStateBuffer(self.addr_bits)

    def SetMaxAddr(self):
        self.latch.SetMaxData()

    def SetAddr(self, addrs):
        self.latch.SetData(addrs)

    def readAddr(self, enable):
        return self.tri(self.latch.getQ(), enable)
        
    def __call__(self, data, clk):
        self.latch(datas=data, clk=clk)


In [40]:
class IncrementerDecrementer:
    def __init__(self):
        """ 
        The Incrementer / Decrementer (pages 350 – 351)
        refer: https://codehiddenlanguage.com/Chapter22/
        """
        self.addr_bits = 16
        self.xor_first_level = [XOR() for _ in range(self.addr_bits)]
        self.xor_second_level = [XOR() for _ in range(self.addr_bits)]
        self.and_gates = [AND() for _ in range(self.addr_bits)]
        self.v = 1

        self.latch = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.addr_bits)
        self.tri = TriStateBuffer(self.addr_bits)
        self.or_gate = OR()

    def __call__(self, addrs, dec, inc, clock):
        
        self.latch(addrs, clock)
        read_addrs = self.latch.getQ()
        
        v = self.v
        output_from_Inc_Dec = []
        for idx, addr in enumerate(reversed(read_addrs)):
            output_from_Inc_Dec.append(self.xor_second_level[idx]([addr, v]))
            v = self.and_gates[idx]([self.xor_first_level[idx]([dec, addr]), v])

        # now LSB is idx 0, so we should reverse, let MSB is idx 0
        output_from_Inc_Dec = list(reversed(output_from_Inc_Dec))

        return self.tri(output_from_Inc_Dec, self.or_gate([dec, inc]))

In [41]:
# %%writefile test_10_inc_dec.py

import sys
sys.path.append("../src")
# from utils import IncrementerDecrementer, int_to_nbit_list, bit_list_to_int

# ==========================================
# 1. BASIC INCREMENT & DECREMENT
# ==========================================
def test_basic_inc_dec():
    """Tests standard +1 and -1 operations."""
    inc_dec = IncrementerDecrementer()
    
    # Test Increment: 5 -> 6
    addr_in = int_to_nbit_list(5, 16)
    # Clock transitions from 0 to 1 to latch the input
    inc_dec(addr_in, dec=0, inc=1, clock=0) 
    out_bits = inc_dec(addr_in, dec=0, inc=1, clock=1)
    
    assert bit_list_to_int(out_bits, signed=False) == 6, "Failed to increment 5 to 6"

    # Test Decrement: 10 -> 9
    addr_in = int_to_nbit_list(10, 16)
    inc_dec(addr_in, dec=1, inc=0, clock=0)
    out_bits = inc_dec(addr_in, dec=1, inc=0, clock=1)
    
    assert bit_list_to_int(out_bits, signed=False) == 9, "Failed to decrement 10 to 9"

test_basic_inc_dec()
# ==========================================
# 2. THE OVERFLOW (ROLLOVER) STRESS TEST
# ==========================================
def test_increment_rollover():
    """Tests the critical hardware boundary: 0xFFFF + 1 must equal 0x0000."""
    inc_dec = IncrementerDecrementer()
    
    # 65535 is 1111111111111111 in binary
    addr_in = int_to_nbit_list(65535, 16)
    
    inc_dec(addr_in, dec=0, inc=1, clock=0)
    out_bits = inc_dec(addr_in, dec=0, inc=1, clock=1)
    
    # The 17th carry bit falls off the edge of the physical silicon. 
    # It must return strictly 0.
    assert bit_list_to_int(out_bits, signed=False) == 0, "Hardware failed to wrap from FFFF to 0000!"

test_increment_rollover()

# ==========================================
# 3. THE UNDERFLOW (REVERSE) STRESS TEST
# ==========================================
def test_decrement_underflow():
    """Tests the reverse boundary: 0x0000 - 1 must equal 0xFFFF."""
    inc_dec = IncrementerDecrementer()
    
    addr_in = int_to_nbit_list(0, 16)
    
    inc_dec(addr_in, dec=1, inc=0, clock=0)
    out_bits = inc_dec(addr_in, dec=1, inc=0, clock=1)
    
    assert bit_list_to_int(out_bits, signed=False) == 65535, "Hardware failed to underflow from 0000 to FFFF!"

test_decrement_underflow()

# ==========================================
# 4. TRI-STATE ISOLATION (BUS DISCONNECT)
# ==========================================
def test_inc_dec_disabled_output():
    """Proves that if neither INC nor DEC is asserted, the component yields the bus."""
    inc_dec = IncrementerDecrementer()
    
    addr_in = int_to_nbit_list(100, 16)
    
    # dec=0, inc=0
    inc_dec(addr_in, dec=0, inc=0, clock=0)
    out_bits = inc_dec(addr_in, dec=0, inc=0, clock=1)
    
    # The output should be 16 zeros (acting as our disconnected Tri-State buffer)
    assert sum(out_bits) == 0, "Tri-state shield failed! Data leaked onto the bus while disabled."

test_inc_dec_disabled_output()

In [25]:
maps = {0: "A", 1: "B",}
maps[0]
x = [0, 0, 0, 0, 0, 1, 0, 0]
idx = x.index(1)
idx

print(maps)
for idx in maps:
    print(maps[idx])
print(len(maps))
print(x[:1])
[2] + x

{0: 'A', 1: 'B'}
A
B
2
[0]


[2, 0, 0, 0, 0, 0, 1, 0, 0]